In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

In [0]:
df = spark.table("workspace.default.capstone_bronze_sales")

df_flagged = df.withColumn(
    "quality_flag",
    F.when(F.col("customer_id").isNull(), "INVALID_CUSTOMER")
     .when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "INVALID_QUANTITY")
     .when(F.col("net_amount").isNull() | (F.col("net_amount") < 0), "INVALID_AMOUNT")
     .when(F.col("product_id").isNull() | (F.upper(F.trim(F.col("product_id"))) == "UNKNOWN"), "INVALID_PRODUCT")
     .otherwise("VALID")
)

In [0]:
total = df_flagged.count()
report = (df_flagged.groupBy("quality_flag")
          .agg(F.count("*").alias("record_count"))
          .withColumn("pct_of_total", F.round(F.col("record_count") / F.lit(total) * 100, 2))
          .orderBy(F.desc("record_count")))

print("===== DATA QUALITY REPORT =====")
report.show(truncate=False)

report.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.capstone_dq_report")